# B2-019 — Practice p17

**Set:** C · **Type:** integrative · **Difficulty:** advanced · **Minutes:** 65

**Concepts:** causal-self-attention, sinusoidal-positional-encoding

**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20260808`  
**Qualified Book 1 prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:C6-pytorch`, `book1:C11-neural-training`  
**Remediation:** review the linked Book 1 units before continuing: [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb).

## Task

Implement the exact one-head `CausalPredictor(nn.Module)` architecture: four bias-free `nn.Linear(4,4,dtype=torch.float64)` projections named `q`, `k`, `v`, and `out`, followed by `head = nn.Linear(4,4,bias=True,dtype=torch.float64)`. In `forward(inputs, allowed)`, compute scaled dot-product attention with scale `sqrt(4)`, apply the Boolean causal mask before softmax, and return logits from the class head. Do not add residuals, normalization, dropout, or another layer.

Use the supplied pinned numeric and one-hot sequence and sinusoidal table exactly: `inputs` is the first five rows of `X + 0.1 * POSITIONAL`, `targets` is the next-symbol class sequence, and `allowed` is lower triangular. Instantiate the model, copy every array in `INITIAL_PARAMETERS` into the same-named parameter before training, and record `logits_before`. Train for exactly 40 full-batch Adam steps at `lr=0.03` using mean cross-entropy reduction over all five target rows, recording each pre-update scalar in `losses`. Then compute `logits_after`, return `probe_logits=torch.stack((logits_before[0,0],logits_after[0,-1]))`, and set `predictions=logits_after.argmax(dim=-1)`. Justify why row i cannot use a token after i while predicting target i.

Pinned contract: seed `20260808`; exact shapes input `(1,5,4)`, targets `(1,5)`, logits `(1,5,4)`, `losses=(40,)`, `probe_logits=(2,4)`, and `predictions=(1,5)`; fixed initialization and first/last probes come from the supplied setup; `atol=1e-8`, `rtol=1e-8`. Allowed APIs: NumPy numeric table construction, `torch`, `nn.Module`, `nn.Linear`, `torch.optim.Adam`, cross-entropy. Banned APIs: learned token lookup, pretrained representations, Transformer convenience modules, CUDA/MPS, file/network access. Reasoning and the deterministic loss trace are required.

In [ ]:
import numpy as np
import torch
from torch import nn

SEED = 20260808
np.random.seed(SEED)
torch.manual_seed(SEED)
X = np.eye(4, dtype=np.float64)[[0, 1, 2, 1, 3, 0]]
positions = np.arange(5, dtype=np.float64)[:, None]
rates = np.array([[1.0, 0.1]], dtype=np.float64)
POSITIONAL = np.empty((5, 4), dtype=np.float64)
POSITIONAL[:, 0::2] = np.sin(positions * rates)
POSITIONAL[:, 1::2] = np.cos(positions * rates)
inputs = torch.tensor(
    X[:-1] + 0.1 * POSITIONAL, dtype=torch.float64, device="cpu"
).unsqueeze(0)
targets = torch.tensor([[1, 2, 1, 3, 0]], dtype=torch.long, device="cpu")
allowed = torch.tril(torch.ones(5, 5, dtype=torch.bool, device="cpu"))
INITIAL_PARAMETERS = {
    "q.weight": np.array(
        [[0.20, -0.10, 0.00, 0.10], [0.00, 0.30, -0.20, 0.10],
         [0.10, 0.00, 0.25, -0.15], [-0.20, 0.10, 0.05, 0.30]], dtype=np.float64
    ),
    "k.weight": np.array(
        [[0.15, 0.00, -0.10, 0.20], [0.10, 0.25, 0.00, -0.10],
         [-0.05, 0.10, 0.30, 0.00], [0.20, -0.15, 0.10, 0.25]], dtype=np.float64
    ),
    "v.weight": np.array(
        [[0.30, 0.00, 0.10, -0.10], [-0.10, 0.20, 0.00, 0.25],
         [0.05, -0.20, 0.35, 0.00], [0.10, 0.15, -0.10, 0.20]], dtype=np.float64
    ),
    "out.weight": np.array(
        [[0.25, -0.05, 0.10, 0.00], [0.00, 0.30, -0.10, 0.05],
         [-0.15, 0.05, 0.20, 0.10], [0.10, -0.10, 0.00, 0.25]], dtype=np.float64
    ),
    "head.weight": np.array(
        [[0.20, 0.00, -0.10, 0.10], [-0.05, 0.25, 0.10, 0.00],
         [0.10, -0.15, 0.30, 0.05], [0.00, 0.10, -0.05, 0.20]], dtype=np.float64
    ),
    "head.bias": np.array([0.01, -0.02, 0.03, -0.04], dtype=np.float64),
}
assert inputs.shape == (1, 5, 4)
assert targets.shape == (1, 5)
assert allowed.shape == (5, 5)

In [ ]:
# Implement CausalPredictor, initialize it, train it, and return the required variables here.


## Your response

Show the requested derivation, implementation, audit, or justification here.